# 03a · RAG question set: Karnataka coffee (needs only GEMINI_API_KEY and GROQ_API_KEY in `.env`)

The benchmark asks what a coffee grower in **Chikkamagaluru, Kodagu or Hassan (Karnataka)** would ask, about
five topics: leaf rust, Cercospora leaf spot, Phoma, leaf miner, and cultivation practice (shade, pruning,
nutrition, soil, irrigation, planting). Every question has a **gold answer** and **gold evidence quotes** copied
word-for-word from a page of the 14 knowledge-base PDFs, so retrieval can be scored the same way for every system.

What runs (fully automatic, ≈ 15-25 min, resumable: every LLM answer is cached):
1. **Preflight**: checks each model answers with your keys; stops with a clear fix if not.
2. Copies the 14 KB PDFs from the app (read-only) into `03_rag/kb/` with a SHA-256 manifest.
3. **Draft**: random KB passages per topic → the JUDGE model writes a grower question, a short answer and the
   exact evidence sentence(s). Drafts whose evidence is not literally in the passage are dropped.
4. **Automatic check**: the JUDGE model rates each draft (answerable from the passage? clear? relevant to a
   Karnataka grower?) and translates it to Kannada. The best 15 per topic are kept; half are asked in Kannada.
5. 10 Karnataka questions whose answer is **not** in the documents (does a system admit it doesn't know?).
6. Freezes `03_rag/data/questions.jsonl` (+ SHA-256).

Optional human review: edit `03_rag/data/questions_review.csv` (status `reject` / `edit`) and re-run this
notebook; the frozen set then records `"reviewed": true`. Not required.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd

from common import config, env
from common import rag_eval as R
from common.llm import LLM, available_models, role_model

config.ensure_dirs()
PER_TOPIC = 3 if config.SMOKE else 15            # questions kept per topic
DRAFT_PER_TOPIC = 4 if config.SMOKE else 22      # drafted per topic (the automatic check keeps the best)
DATA = config.RAG_DATA
DRAFT, REVIEW, FROZEN = DATA / "questions_draft.csv", DATA / "questions_review.csv", DATA / "questions.jsonl"

## 1. Preflight: do the keys and models work?

In [ ]:
roles = {r: role_model(r, verbose=True) for r in ("GENERATOR", "BASELINE_LLM", "JUDGE")}
problems = []
for role, spec in roles.items():
    try:
        err = LLM(spec).ping()
    except Exception as exc:  # noqa: BLE001
        err = str(exc)
    print(f"{role:13s} {spec:40s} {'OK' if not err else 'FAILED: ' + err[:200]}")
    if err:
        provider = spec.split(":")[0]
        names = available_models(provider)
        hint = [n for n in names if re.search(r"flash|llama|gpt-oss|qwen", n)][:15]
        problems.append(f"{role}: {spec} does not work ({err[:150]}).\n   Fix: put a working model in compare/.env, "
                        f"e.g. {role}_MODEL={provider}:<model>. Models your key can use: {hint or names[:15]}")
if problems:
    raise SystemExit("\n".join(problems))
print("\nAll models answer. Roles:", roles)

## 2. Knowledge base (copied once from the app, never modified)

In [ ]:
src_pdfs = sorted(config.APP_KB_DIR.glob("*.pdf"))
assert src_pdfs, f"no PDFs in {config.APP_KB_DIR}"
for p in src_pdfs:
    dst = config.KB_DIR / p.name
    if not dst.exists():
        shutil.copy2(p, dst)
manifest = pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size,
                          "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(config.KB_DIR.glob("*.pdf"))])
manifest.to_csv(config.KB_DIR / "kb_manifest.csv", index=False)
pages = R.page_texts()
print(f"{len(manifest)} PDFs, {len(pages)} pages with text")

## 3. Draft questions from random passages, then check them automatically

In [ ]:
def passages(topic, words=140):
    keys = R.TOPIC_KEYWORDS[topic]
    out = []
    for pg in pages:
        sents = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", pg["text"]).strip())
        buf = []
        for s in sents:
            buf.append(s)
            if sum(len(x.split()) for x in buf) >= words:
                text = " ".join(buf)
                # keyword present and mostly real words (skips pages whose PDF text is garbled)
                letters = sum(c.isalpha() for c in text) / max(1, len(text))
                if any(k in text.lower() for k in keys) and len(text) > 300 and letters > 0.7:
                    out.append({"source": pg["source"], "page": pg["page"], "text": text})
                buf = []
    return out

DRAFT_PROMPT = """You are building an evaluation set for an advice system used by coffee growers in Karnataka, India
(Chikkamagaluru, Kodagu and Hassan districts; Arabica and Robusta estates).
Read the passage from an agronomy document and write ONE question that such a grower might really ask and that this
passage answers. Topic: {topic}. The question must make sense on its own (never mention "the passage", "the
document" or "the study"), be practical (what to do, when, how much, how to recognise), and not be a trivia question
about authors, tables or statistics.

<passage>
{passage}
</passage>

Return JSON {{"questions": [{{"question": "...", "answer": "short correct answer (1-3 sentences) based only on the passage",
"evidence": ["1-2 sentences copied EXACTLY, word for word, from the passage that contain the answer"]}}]}}"""

CHECK_PROMPT = """Check a draft evaluation question for a coffee advice system used by growers in Karnataka, India.
QUESTION: {q}
GOLD ANSWER: {a}
EVIDENCE FROM THE DOCUMENT: {e}

Return JSON {{"answerable": 1 if the evidence fully answers the question else 0,
"clear": 1 if the question is clear and self-contained else 0,
"grower_relevance": 1-5 (5 = a practical question a Karnataka coffee grower would really ask),
"kannada": "the question translated into natural, simple Kannada (Kannada script); keep numbers, units and chemical names"}}"""

OOK = ["What is today's coffee auction price at the Coffee Board in Bengaluru?",
       "Which drone model is best for spraying coffee estates in Kodagu?",
       "How much subsidy does the Coffee Board give for replanting this year?",
       "How do I register my Chikkamagaluru estate to export coffee to Japan?",
       "What is the interest rate on crop loans for coffee growers at SBI?",
       "How many litres of diesel does a tractor use per acre of coffee?",
       "Which mobile network has the best coverage in Chikkamagaluru estates?",
       "What daily wage should I pay coffee pickers in Hassan this season?",
       "How do I treat foot-and-mouth disease in cattle grazing in my estate?",
       "Can I grow saffron between my coffee rows in Kodagu?"]

judge = LLM(roles["JUDGE"])
if DRAFT.exists():
    draft = pd.read_csv(DRAFT, encoding="utf-8-sig")
    print(f"re-using {DRAFT.name} ({len(draft)} rows); delete it to redraft")
else:
    rng = np.random.default_rng(config.SEED)
    rows = []
    for topic in R.TOPICS:
        pool = passages(topic)
        by_src = {}
        for p in pool:
            by_src.setdefault(p["source"], []).append(p)
        for v in by_src.values():
            rng.shuffle(v)
        picked, i = [], 0     # round-robin over PDFs so no single document dominates
        while len(picked) < DRAFT_PER_TOPIC and any(by_src.values()):
            src = list(by_src)[i % len(by_src)]
            if by_src[src]:
                picked.append(by_src[src].pop())
            i += 1
        print(f"{topic}: {len(pool)} candidate passages from {len(by_src)} PDFs -> drafting {len(picked)}")
        for p in picked:
            d, _ = judge.generate_json(DRAFT_PROMPT.format(topic=R.TOPIC_NAMES[topic], passage=p["text"]),
                                       temperature=0.3)
            q = (d.get("questions") or [{}])[0]
            ev = [e for e in q.get("evidence", []) if R.quote_in_text(e, p["text"]) >= 0.95]
            if not ev or not q.get("question"):
                continue
            c, _ = judge.generate_json(CHECK_PROMPT.format(q=q["question"], a=q.get("answer", ""), e=" ".join(ev)))
            rows.append({"topic": topic, "disease": R.TOPICS[topic], "question_en": q["question"].strip(),
                         "question_kn": str(c.get("kannada", "")).strip(),
                         "gold_answer": q.get("answer", "").strip(), "evidence": json.dumps(ev, ensure_ascii=False),
                         "source": p["source"], "page": p["page"], "out_of_kb": 0,
                         "answerable": int(c.get("answerable", 0) or 0), "clear": int(c.get("clear", 0) or 0),
                         "grower_relevance": int(c.get("grower_relevance", 0) or 0)})
    cand = pd.DataFrame(rows)
    cand["score"] = cand["answerable"] * 10 + cand["clear"] * 5 + cand["grower_relevance"]
    keep = []
    for topic, d in cand.groupby("topic", sort=False):
        ok = d[(d.answerable == 1) & (d.clear == 1)].sort_values("score", ascending=False, kind="stable")
        sel = ok.head(PER_TOPIC).copy()
        sel["lang"] = ["kn" if (j % 2 == 0 and q) else "en" for j, q in enumerate(sel["question_kn"])]
        keep.append(sel)
        print(f"{topic}: {len(d)} drafts -> {len(ok)} passed the check -> {len(sel)} kept")
    draft = pd.concat(keep, ignore_index=True)
    draft["question"] = np.where(draft["lang"] == "kn", draft["question_kn"], draft["question_en"])
    ook = pd.DataFrame([{"topic": "cultivation", "disease": "Healthy", "question_en": q, "question_kn": "",
                         "question": q, "lang": "en", "gold_answer": "", "evidence": "[]", "source": "", "page": "",
                         "out_of_kb": 1} for q in OOK])
    draft = pd.concat([draft, ook], ignore_index=True)
    draft.insert(0, "id", [f"q{i:03d}" for i in range(len(draft))])
    draft["status"] = "auto"
    draft["reviewer_note"] = ""
    draft.to_csv(DRAFT, index=False, encoding="utf-8-sig")      # utf-8-sig: Excel shows Kannada correctly
    print(f"wrote {DRAFT.name} ({len(draft)} rows)")
if not REVIEW.exists():
    shutil.copy2(DRAFT, REVIEW)
draft.groupby(["topic", "lang"]).size().unstack(fill_value=0)

## 4. Freeze the question set
Uses `questions_review.csv`: rows with status `reject` are dropped, `edit` rows use your edited text; everything
else (`auto`, `accept`) is kept. Evidence must stay a word-for-word copy from the cited PDF page.

In [ ]:
rev = pd.read_csv(REVIEW, encoding="utf-8-sig", dtype=str).fillna("")
rev["status"] = rev["status"].str.strip().str.lower().replace("", "auto")
human = bool(rev["status"].isin(["accept", "edit", "reject", "new"]).any())
keep = rev[rev.status != "reject"].copy()
page_text = {(p["source"], str(p["page"])): p["text"] for p in pages}
problems, records = [], []
for _, r in keep.iterrows():
    ook = r["out_of_kb"] in ("1", "1.0", "True", "true")
    ev = json.loads(r["evidence"] or "[]")
    if not ook:
        text = page_text.get((r["source"], str(r["page"]).split(".")[0]))
        if text is None or not ev or any(R.quote_in_text(e, text) < 0.95 for e in ev):
            problems.append((r["id"], f"evidence not found word-for-word on {r['source']} p.{r['page']}"))
            continue
    records.append({"id": r["id"], "topic": r["topic"], "disease": R.TOPICS[r["topic"]], "lang": r["lang"] or "en",
                    "question": r["question"].strip(), "question_en": r["question_en"].strip(),
                    "gold_answer": r["gold_answer"].strip(), "evidence": ev, "source": r["source"],
                    "page": int(float(r["page"])) if r["page"] else None, "out_of_kb": int(ook),
                    "status": r["status"]})
for p in problems:
    print("dropped (fix in questions_review.csv to keep):", *p)

with FROZEN.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
digest = hashlib.sha256(FROZEN.read_bytes()).hexdigest()
(DATA / "questions.jsonl.sha256").write_text(f"{digest}  questions.jsonl\n")
qs = pd.DataFrame(records)
env.write_json(DATA / "questions_meta.json", {
    "n": len(qs), "out_of_kb": int(qs.out_of_kb.sum()), "reviewed": human,
    "selection": "LLM-drafted from KB passages, automatically checked (answerable, clear, grower relevance); "
                 + ("then reviewed by the team" if human else "not human-reviewed"),
    "by_topic_lang": qs.groupby(["topic", "lang"]).size().unstack(fill_value=0).to_dict(),
    "models": roles, "sha256": digest, "meta": env.run_meta()})
print(f"FROZEN {len(qs)} questions ({int(qs.out_of_kb.sum())} out of KB) -> {FROZEN.name}  sha256 {digest[:16]}...")
qs.groupby(["topic", "lang"]).size().unstack(fill_value=0)